# Topic: SQL Pivot Table Pattern

## Definition (30-second explanation)
* The Pivot operation transforms rows into columns, reshaping data from a long/tall format to a wide format.
* Because most databases (like MySQL and PostgreSQL) lack a native `PIVOT` keyword, the standard approach is conditional aggregation: using `CASE WHEN` inside aggregation functions like `SUM()`, `COUNT()`, or `MAX()`.

## Why Interviewers Ask This
* Pivoting is a fundamental data preparation step for dashboards, BI tools (Tableau, PowerBI), and spreadsheet exports.
* It tests your understanding of the `GROUP BY` clause and how aggregation functions evaluate row-level logic.

## Core Concepts
* **Conditional Aggregation:** The core engine of a pivot. E.g., `SUM(CASE WHEN category = 'A' THEN value ELSE 0 END)`.
* **Row Identifier:** The column(s) placed in the `GROUP BY` clause become the unique row identifiers in the final wide table.
* **Native vs Universal:** SQL Server and Oracle have a native `PIVOT()` operator, but the `CASE WHEN` approach is universal across almost all relational databases.

## When to Use
* Converting monthly or quarterly metric rows into side-by-side columns for management reports.
* Turning category-value (EAV model) pairs into dedicated columns for machine learning features.
* Reshaping survey response data (one row per answer) into one row per respondent.
* Creating crosstab/matrix reports.

## Advantages
* The `CASE WHEN` pivot is universally supported and highly readable.
* Allows for complex transformations during the pivot (e.g., calculating YoY growth between the newly created columns in the same query).

## Limitations
* **Hardcoded Columns:** Standard SQL pivots require you to know all possible column values at query-writing time. 
* **Dynamic Pivoting:** If the column values change dynamically (e.g., new product names added daily), standard SQL cannot adapt automatically; you must use dynamic SQL (constructing query strings programmatically).

## Common Comparisons
* **Pivot vs. Unpivot:** Pivot turns rows into columns (long to wide). Unpivot (See 'Unpivot Pattern') turns columns into rows (wide to long).
* **`SUM()` vs `MAX()` in Pivots:** Use `SUM()` when aggregating numerical metrics (like revenue). Use `MAX()` when pivoting categorical/string data (like survey answers) to pull the single text value into the column.

## Common Interview Traps
* **Missing the Row Identifier:** Forgetting to include the identifier (e.g., `department`) in the `GROUP BY` clause collapses the entire dataset into a single row.
* **Using `ELSE NULL` instead of `ELSE 0`:** When pivoting numerical sums, omitting the `ELSE 0` (or explicitly using `ELSE NULL`) will return NULLs for missing categories. This can break downstream mathematical operations.
* **Division by Zero:** When calculating growth between pivot columns, an empty period will evaluate to 0, causing a division by zero error. Always wrap the denominator in `NULLIF(denominator, 0)`.

## Python / SQL Syntax
```sql
    -- Standard Universal Pivot (MySQL, Postgres, BigQuery)
    SELECT 
        department,
        SUM(CASE WHEN quarter = 'Q1' THEN revenue ELSE 0 END) AS Q1_revenue,
        SUM(CASE WHEN quarter = 'Q2' THEN revenue ELSE 0 END) AS Q2_revenue,
        SUM(CASE WHEN quarter = 'Q3' THEN revenue ELSE 0 END) AS Q3_revenue,
        SUM(CASE WHEN quarter = 'Q4' THEN revenue ELSE 0 END) AS Q4_revenue,
        SUM(revenue) AS total_annual_revenue
    FROM sales
    GROUP BY department;
```

## 45-Second Interview Answer
"To pivot data from a long to a wide format in standard SQL, I use conditional aggregation. I group by the desired row identifier, and for each new column, I wrap a `CASE WHEN` statement inside an aggregate function like `SUM` or `MAX`. This evaluates each row, placing the value into the correct column and defaulting to 0 or NULL for the rest. While SQL Server has a native PIVOT operator, this `CASE WHEN` method is universal. The main limitation is that you must hardcode the columns; for fully dynamic pivoting, I would need to write a stored procedure using dynamic SQL or handle it in the BI/Pandas layer."

## Example Questions:

### Q1: Pivot monthly sales data (Jan-Dec) into columns. Also calculate the month with the highest sales using MAX() and CASE WHEN.

* **Ideal Interview Answer:** I'd group by the relevant dimension (e.g., year or store) and write 12 `SUM(CASE WHEN month = 'Jan'...)` statements. 
```sql
    SELECT store_id,
           SUM(CASE WHEN month = 'Jan' THEN sales ELSE 0 END) AS Jan_sales,
           SUM(CASE WHEN month = 'Feb' THEN sales ELSE 0 END) AS Feb_sales,
           -- ... (Mar through Nov)
           SUM(CASE WHEN month = 'Dec' THEN sales ELSE 0 END) AS Dec_sales,
           MAX(sales) AS highest_monthly_sales
    FROM monthly_sales
    GROUP BY store_id;
```
* **Common Mistakes:** Using `ELSE NULL` for numerical sales data instead of `ELSE 0`.
* **Likely Interviewer Follow-up:** The prompt asked for the *month* with the highest sales, but `MAX(sales)` just returns the highest *value*. How would you get the actual month name? (Answer: I would need a CTE with a window function like `RANK() OVER(PARTITION BY store_id ORDER BY sales DESC)` to flag the top month, and then pivot that alongside the data, or use a complex `GREATEST` comparison across the pivoted columns).

### Q2: Transform a survey table (user_id, question_id, answer) into one row per user with columns for each question.

* **Ideal Interview Answer:** Because survey answers are typically text/strings, I cannot use `SUM()`. Instead, I will use `MAX()` as the aggregation function to pull the string value into the pivoted column.
```sql
    SELECT user_id,
           MAX(CASE WHEN question_id = 1 THEN answer END) AS question_1_answer,
           MAX(CASE WHEN question_id = 2 THEN answer END) AS question_2_answer,
           MAX(CASE WHEN question_id = 3 THEN answer END) AS question_3_answer
    FROM survey_responses
    GROUP BY user_id;
```
* **Common Mistakes:** Trying to use `SUM()` on text columns, which will result in an error or `0`.
* **Likely Interviewer Follow-up:** Why does `MAX()` work for strings here? (Answer: When grouped by `user_id`, there is only one answer per `question_id`. The `CASE WHEN` makes all other rows NULL for that specific column. `MAX()` ignores NULLs and simply returns the single non-NULL string value).

### Q3: Create a report showing, for each product category, how many products are in price ranges: under 100, 100-500, 500-1000, over 1000.

* **Ideal Interview Answer:** I will group by product category and use conditional counting. I prefer using `SUM` with 1 and 0 for this type of bucketing pivot.
```sql
    SELECT category,
           SUM(CASE WHEN price < 100 THEN 1 ELSE 0 END) AS under_100,
           SUM(CASE WHEN price >= 100 AND price < 500 THEN 1 ELSE 0 END) AS range_100_to_500,
           SUM(CASE WHEN price >= 500 AND price <= 1000 THEN 1 ELSE 0 END) AS range_500_to_1000,
           SUM(CASE WHEN price > 1000 THEN 1 ELSE 0 END) AS over_1000
    FROM products
    GROUP BY category;
```
* **Common Mistakes:** Using `COUNT(CASE WHEN price < 100 THEN 1 ELSE 0 END)`. `COUNT()` counts non-nulls, so it would count the `0`s as well. To use `COUNT`, you must remove the `ELSE 0` so it defaults to NULL.
* **Likely Interviewer Follow-up:** How would you ensure all categories appear in the report, even if they have zero products in these ranges? (Answer: Ensure the `products` table is `LEFT JOIN`ed from a master `categories` table before doing the pivot aggregation).

### Q4: Pivot attendance data to show for each employee: days_present, days_absent, days_on_leave as separate columns.

* **Ideal Interview Answer:** Similar to the previous question, this is a conditional counting pivot. I will group by employee and sum the occurrences of each status.
```sql
    SELECT employee_id,
           SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) AS days_present,
           SUM(CASE WHEN status = 'Absent' THEN 1 ELSE 0 END) AS days_absent,
           SUM(CASE WHEN status = 'Leave' THEN 1 ELSE 0 END) AS days_on_leave
    FROM attendance
    GROUP BY employee_id;
```
* **Common Mistakes:** Forgetting the `GROUP BY employee_id`, resulting in a single row showing the total days present/absent for the entire company.
* **Likely Interviewer Follow-up:** How would you add a column for the employee's attendance percentage? (Answer: I would calculate `SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) / COUNT(*) * 100.0` within the same `SELECT` statement).